# SDXL × DrawBench — ImageReward Scoring (Colab)

Very annoying, but I could not figure out to run IR without having its seperate environment.

In [1]:
# Pin transformers (v4 — ImageReward's BLIP needs helpers removed in v5)
# and diffusers (ImageReward's ReFL.py imports diffusers at module level).
# diffusers 0.30.3: >=0.29 (no cached_download), predates AutoencoderRAE
# which needs transformers v5. 0.30.3 requires transformers>=4.41.2 — OK.
!pip install -q transformers==4.44.2 diffusers==0.30.3 accelerate torch torchvision pillow pandas
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
# wandb is pre-installed on Colab but its protobuf files are broken;
# ImageReward's ReFL.py imports wandb if present, causing a crash.
!pip uninstall -y wandb

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA L4


## 1. Data Preparation

In [4]:
DRIVE_DIR = Path("/content/drive/MyDrive/generative-modelling")

DATA_DIR = Path("/content/data")
STANDARD_DIR = DATA_DIR / "sdxl_drawbench"
ZSAMPLING_DIR = DATA_DIR / "sdxl_drawbench_zsampling"

SEEDS = [42, 123, 2026]
NUM_PROMPTS = 200

In [5]:
import shutil
import time

DATA_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
for name in ["sdxl_drawbench.zip", "sdxl_drawbench_zsampling.zip"]:
    src = DRIVE_DIR / name
    dst = DATA_DIR / name
    if not dst.exists():
        print(f"Copying {name} ({src.stat().st_size / 1e6:.0f} MB)...")
        shutil.copy2(src, dst)
    else:
        print(f"{name} already copied.")

for name in ["sdxl_drawbench.zip", "sdxl_drawbench_zsampling.zip"]:
    target = DATA_DIR / name.replace(".zip", "")
    if not target.exists():
        print(f"Extracting {name}...")
        with zipfile.ZipFile(DATA_DIR / name) as z:
            z.extractall(DATA_DIR)
    else:
        print(f"{name.replace('.zip', '')}/ already extracted.")

print(f"\nDone in {time.time() - t0:.1f}s")
print(f"Standard: {STANDARD_DIR}")
print(f"Z-Sampling: {ZSAMPLING_DIR}")

sdxl_drawbench.zip already copied.
sdxl_drawbench_zsampling.zip already copied.
sdxl_drawbench/ already extracted.
sdxl_drawbench_zsampling/ already extracted.

Done in 0.0s
Standard: /content/data/sdxl_drawbench
Z-Sampling: /content/data/sdxl_drawbench_zsampling


In [6]:
def index_images(base_dir: Path) -> dict[str, Path]:
    """Build a filename -> path index across all checkpoint subfolders."""
    index = {}
    for png in base_dir.rglob("*.png"):
        index[png.name] = png
    return index

std_index = index_images(STANDARD_DIR)
zs_index = index_images(ZSAMPLING_DIR)

print(f"Standard images found: {len(std_index)} (expected 600)")
print(f"Z-Sampling images found: {len(zs_index)} (expected 600)")
assert len(std_index) == 600, f"Expected 600 standard images, got {len(std_index)}"
assert len(zs_index) == 600, f"Expected 600 z-sampling images, got {len(zs_index)}"

Standard images found: 600 (expected 600)
Z-Sampling images found: 600 (expected 600)


## 2. Load DrawBench Prompts & Build Pairs

In [7]:
CSV_URL = "https://huggingface.co/datasets/sayakpaul/drawbench/raw/main/DrawBench%20Prompts%20-%20Sheet1.csv"
df_prompts = pd.read_csv(CSV_URL)
PROMPTS = df_prompts["Prompts"].tolist()
assert len(PROMPTS) == NUM_PROMPTS, f"Expected {NUM_PROMPTS} prompts, got {len(PROMPTS)}"
print(f"Loaded {len(PROMPTS)} DrawBench prompts")

Loaded 200 DrawBench prompts


In [8]:
def build_pairs() -> pd.DataFrame:
    """Build a DataFrame of (prompt_idx, seed, prompt, standard_path, zsampling_path)."""
    rows = []
    missing = []
    for seed in SEEDS:
        for idx in range(NUM_PROMPTS):
            std_name = f"sdxl_prompt{idx:03d}_seed{seed}.png"
            zs_name = f"sdxl_zsample_prompt{idx:03d}_seed{seed}.png"
            std_path = std_index.get(std_name)
            zs_path = zs_index.get(zs_name)
            if std_path is None or zs_path is None:
                missing.append((idx, seed, std_name if std_path is None else zs_name))
                continue
            rows.append({
                "prompt_idx": idx,
                "seed": seed,
                "prompt": PROMPTS[idx],
                "standard_path": str(std_path),
                "zsampling_path": str(zs_path),
            })
    if missing:
        print(f"WARNING: {len(missing)} missing images!")
        for idx, seed, name in missing[:5]:
            print(f"  prompt={idx} seed={seed}: {name}")
    else:
        print("All 600 pairs verified.")
    return pd.DataFrame(rows)

df_pairs = build_pairs()
print(f"Total pairs: {len(df_pairs)}")

All 600 pairs verified.
Total pairs: 600


## 3. Load ImageReward

In [9]:
import ImageReward as RM

ir_model = RM.load("ImageReward-v1.0", device=DEVICE)
print("ImageReward model loaded.")

ImageReward.pt:   0%|          | 0.00/1.79G [00:00<?, ?B/s]

load checkpoint from /root/.cache/ImageReward/ImageReward.pt


med_config.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

checkpoint loaded
ImageReward model loaded.


## 4. Scoring

In [10]:
from tqdm.auto import tqdm

ir_results = []
wins_zs = 0
t0 = time.time()

bar = tqdm(df_pairs.iterrows(), total=len(df_pairs), desc="ImageReward", unit="pair")
for i, (_, row) in enumerate(bar):
    prompt = row["prompt"]
    ir_std = float(ir_model.score(prompt, row["standard_path"]))
    ir_zs  = float(ir_model.score(prompt, row["zsampling_path"]))

    ir_results.append({
        "prompt_idx": int(row["prompt_idx"]),
        "seed": int(row["seed"]),
        "ir_standard": ir_std,
        "ir_zsampling": ir_zs,
    })

    if ir_zs > ir_std:
        wins_zs += 1
    bar.set_postfix(zs_win=f"{wins_zs/(i+1)*100:.0f}%")

    if (i + 1) % 50 == 0:
        tqdm.write(
            f"  [{i+1:>4}/{len(df_pairs)}] "
            f"last: std={ir_std:.3f} zs={ir_zs:.3f} | "
            f"Z-win rate so far: {wins_zs/(i+1)*100:.1f}%"
        )

elapsed = time.time() - t0
print(f"\nDone. Scored {len(df_pairs)} pairs in {elapsed/60:.1f} min")

df_ir = pd.DataFrame(ir_results)
print(f"IR mean — Standard: {df_ir['ir_standard'].mean():.4f}, Z-Sampling: {df_ir['ir_zsampling'].mean():.4f}")
print(f"Z-Sampling wins: {wins_zs}/{len(df_pairs)} ({wins_zs/len(df_pairs)*100:.1f}%)")

ImageReward:   0%|          | 0/600 [00:00<?, ?pair/s]

  [  50/600] last: std=1.071 zs=1.136 | Z-win rate so far: 66.0%
  [ 100/600] last: std=-1.294 zs=-1.227 | Z-win rate so far: 70.0%
  [ 150/600] last: std=1.729 zs=1.611 | Z-win rate so far: 66.7%
  [ 200/600] last: std=1.577 zs=1.320 | Z-win rate so far: 70.0%
  [ 250/600] last: std=1.597 zs=1.306 | Z-win rate so far: 70.8%
  [ 300/600] last: std=-0.981 zs=-1.500 | Z-win rate so far: 68.7%
  [ 350/600] last: std=1.905 zs=1.889 | Z-win rate so far: 67.1%
  [ 400/600] last: std=1.007 zs=1.914 | Z-win rate so far: 67.5%
  [ 450/600] last: std=0.727 zs=1.505 | Z-win rate so far: 66.7%
  [ 500/600] last: std=1.658 zs=1.627 | Z-win rate so far: 67.6%
  [ 550/600] last: std=1.819 zs=1.737 | Z-win rate so far: 66.9%
  [ 600/600] last: std=-0.187 zs=0.574 | Z-win rate so far: 66.8%

Done. Scored 600 pairs in 1.8 min
IR mean — Standard: 0.5908, Z-Sampling: 0.8234
Z-Sampling wins: 401/600 (66.8%)


## 5. Merge IR Scores into existing Results

In [11]:
import json

RESULTS_PATH = DRIVE_DIR / "sdxl_drawbench_eval_results.jsonl"
assert RESULTS_PATH.exists(), f"Results file not found: {RESULTS_PATH}"

with open(RESULTS_PATH) as f:
    records = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(records)} existing records from JSONL.")

ir_lookup = {(r["prompt_idx"], r["seed"]): r for r in ir_results}

updated = 0
for rec in records:
    key = (rec["prompt_idx"], rec["seed"])
    if key in ir_lookup:
        rec["ir_standard"] = ir_lookup[key]["ir_standard"]
        rec["ir_zsampling"] = ir_lookup[key]["ir_zsampling"]
        updated += 1

with open(RESULTS_PATH, "w") as f:
    for rec in records:
        f.write(json.dumps(rec) + "\n")

print(f"Updated {updated}/{len(records)} records with IR scores.")
print(f"Written to: {RESULTS_PATH}")

Loaded 600 existing records from JSONL.
Updated 600/600 records with IR scores.
Written to: /content/drive/MyDrive/generative-modelling/sdxl_drawbench_eval_results.jsonl
